# Quickstart perturbation report

This tutorial creates a small synthetic dataset at runtime and builds a descriptive perturbation report. It is intentionally lightweight and does not access or rerun manuscript benchmark outputs. The report ranks state shifts and exports a figure, source table, deterministic alt text, and metadata.

In [ ]:
from __future__ import annotations

import importlib.metadata as importlib_metadata
import json
import os
import platform
import subprocess
import sys
from datetime import datetime, timezone
from pathlib import Path

os.environ.setdefault("MPLCONFIGDIR", str(Path("/tmp") / "scgeo_revision_matplotlib"))
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

plt.rcParams.update({
    "figure.dpi": 120,
    "savefig.dpi": 220,
    "font.size": 10,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.18,
})


def find_repo_root(start: Path | None = None) -> Path:
    path = (start or Path.cwd()).resolve()
    for candidate in [path, *path.parents]:
        if (candidate / "configs" / "manuscript_benchmark_v1.json").exists():
            return candidate
    raise RuntimeError("Could not locate repository root")


REPO_ROOT = find_repo_root()
CONFIG = json.loads((REPO_ROOT / "configs" / "manuscript_benchmark_v1.json").read_text(encoding="utf-8"))


def resolve_config_path(value: str) -> Path:
    path = Path(value).expanduser()
    if not path.is_absolute():
        path = REPO_ROOT / path
    return path.resolve()


OUTPUT_DIR = resolve_config_path(os.environ.get(CONFIG["notebook_output_env"], CONFIG["default_output_dir"]))
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


def rel_display(path: Path) -> str:
    path = path.resolve()
    try:
        return path.relative_to(REPO_ROOT).as_posix()
    except ValueError:
        return path.as_posix()


def package_version(name: str) -> str:
    try:
        return importlib_metadata.version(name)
    except importlib_metadata.PackageNotFoundError:
        return "not-installed"


def git_commit(path: Path) -> str | None:
    if not (path / ".git").exists():
        return None
    try:
        return subprocess.check_output(["git", "-C", str(path), "rev-parse", "HEAD"], text=True).strip()
    except Exception:
        return None


def write_figure_bundle(stem: str, fig: plt.Figure, source_table: pd.DataFrame, alt_text: str) -> pd.DataFrame:
    fig_dir = OUTPUT_DIR / "figures"
    source_dir = OUTPUT_DIR / "figure_sources"
    alt_dir = OUTPUT_DIR / "alt_text"
    for directory in (fig_dir, source_dir, alt_dir):
        directory.mkdir(parents=True, exist_ok=True)
    png_path = fig_dir / f"{stem}.png"
    svg_path = fig_dir / f"{stem}.svg"
    csv_path = source_dir / f"{stem}.csv"
    alt_path = alt_dir / f"{stem}.txt"
    source_table.to_csv(csv_path, index=False)
    fig.savefig(svg_path, format="svg", bbox_inches="tight", metadata={"Date": None})
    fig.savefig(png_path, format="png", bbox_inches="tight", metadata={"Software": "matplotlib"})
    alt_path.write_text(alt_text.strip() + "\n", encoding="utf-8")
    plt.close(fig)
    return pd.DataFrame([{ 
        "figure": stem,
        "png": rel_display(png_path),
        "svg": rel_display(svg_path),
        "source_csv": rel_display(csv_path),
        "alt_text": rel_display(alt_path),
    }])


def write_metadata(stem: str, extra: dict[str, object] | None = None) -> pd.DataFrame:
    metadata_dir = OUTPUT_DIR / "metadata"
    metadata_dir.mkdir(parents=True, exist_ok=True)
    source_repo = resolve_config_path(os.environ.get(CONFIG["source_repository_env"], CONFIG["default_source_repository"]))
    metadata = {
        "timestamp_utc": datetime.now(timezone.utc).isoformat(),
        "python": sys.version,
        "python_executable": sys.executable,
        "platform": platform.platform(),
        "packages": {name: package_version(name) for name in ["scgeo", "pandas", "numpy", "matplotlib", "nbformat", "nbclient"]},
        "notebook_repository_commit": git_commit(REPO_ROOT),
        "source_commit_expected": CONFIG["expected_source_commit"],
        "source_repository_commit": git_commit(source_repo),
        "protocol_version": CONFIG["protocol_version"],
        "profile": CONFIG["profile"],
        "output_dir": str(OUTPUT_DIR),
        "tutorial_runtime_constraint": "designed for less than five minutes on a laptop kernel",
        "threshold_policy": "descriptive ranking only; no threshold tuning",
    }
    if extra:
        metadata.update(extra)
    path = metadata_dir / f"{stem}_metadata.json"
    path.write_text(json.dumps(metadata, indent=2, sort_keys=True) + "\n", encoding="utf-8")
    return pd.DataFrame([{"metadata": rel_display(path)}])

In [ ]:
rng = np.random.default_rng(20260717)
states = ["state_0", "state_1", "state_2", "state_3"]
conditions = ["control", "treated"]
samples_per_condition = 3
cells_per_state_sample = 45
base_centers = {
    "state_0": np.array([0.0, 0.0]),
    "state_1": np.array([2.2, 0.2]),
    "state_2": np.array([0.2, 2.1]),
    "state_3": np.array([2.3, 2.2]),
}
true_shifts = {
    "state_0": np.array([0.05, 0.00]),
    "state_1": np.array([0.72, 0.18]),
    "state_2": np.array([0.08, -0.03]),
    "state_3": np.array([-0.46, 0.52]),
}
rows = []
for condition in conditions:
    for sample_idx in range(samples_per_condition):
        sample = f"{condition}_{sample_idx}"
        sample_offset = rng.normal(0, 0.05, size=2)
        for state in states:
            n_cells = cells_per_state_sample + (12 if condition == "treated" and state == "state_3" else 0)
            center = base_centers[state] + sample_offset
            if condition == "treated":
                center = center + true_shifts[state]
            coords = rng.normal(center, 0.18, size=(n_cells, 2))
            for x, y in coords:
                rows.append({"condition": condition, "sample": sample, "state": state, "x": x, "y": y})

cells = pd.DataFrame(rows)
cells.head()

In [ ]:
centroids = (
    cells.groupby(["condition", "sample", "state"], as_index=False)
    .agg(x_mean=("x", "mean"), y_mean=("y", "mean"), n_cells=("x", "size"))
)
condition_centroids = (
    centroids.groupby(["condition", "state"], as_index=False)
    .agg(x_mean=("x_mean", "mean"), y_mean=("y_mean", "mean"), n_cells=("n_cells", "sum"))
)
wide = condition_centroids.pivot(index="state", columns="condition", values=["x_mean", "y_mean", "n_cells"])
report = pd.DataFrame({
    "state": states,
    "dx": [wide.loc[state, ("x_mean", "treated")] - wide.loc[state, ("x_mean", "control")] for state in states],
    "dy": [wide.loc[state, ("y_mean", "treated")] - wide.loc[state, ("y_mean", "control")] for state in states],
    "control_cells": [int(wide.loc[state, ("n_cells", "control")]) for state in states],
    "treated_cells": [int(wide.loc[state, ("n_cells", "treated")]) for state in states],
})
report["shift_magnitude"] = np.sqrt(report["dx"] ** 2 + report["dy"] ** 2)
report["rank"] = report["shift_magnitude"].rank(ascending=False, method="first").astype(int)
report = report.sort_values("rank")
report

In [ ]:
# Bootstrap sample labels only; this is a quick descriptive uncertainty calculation, not threshold tuning.
boot_rows = []
for state in states:
    control = centroids[(centroids["condition"] == "control") & (centroids["state"] == state)][["x_mean", "y_mean"]].to_numpy()
    treated = centroids[(centroids["condition"] == "treated") & (centroids["state"] == state)][["x_mean", "y_mean"]].to_numpy()
    magnitudes = []
    for _ in range(500):
        control_sample = control[rng.integers(0, len(control), size=len(control))].mean(axis=0)
        treated_sample = treated[rng.integers(0, len(treated), size=len(treated))].mean(axis=0)
        magnitudes.append(float(np.linalg.norm(treated_sample - control_sample)))
    boot_rows.append({
        "state": state,
        "ci95_low": float(np.quantile(magnitudes, 0.025)),
        "ci95_high": float(np.quantile(magnitudes, 0.975)),
    })
uncertainty = pd.DataFrame(boot_rows)
report = report.merge(uncertainty, on="state", how="left")
report

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10.4, 4.2), constrained_layout=True)
colors = {"control": "#4C78A8", "treated": "#F58518"}
for condition, subset in cells.groupby("condition"):
    axes[0].scatter(subset["x"], subset["y"], s=5, alpha=0.18, color=colors[condition], label=condition.capitalize())
for _, row in condition_centroids.iterrows():
    axes[0].scatter(row["x_mean"], row["y_mean"], s=62, color=colors[row["condition"]], edgecolor="#222222", linewidth=0.6)
axes[0].set_title("Synthetic cells and condition centroids")
axes[0].set_xlabel("dimension 1")
axes[0].set_ylabel("dimension 2")
axes[0].legend(frameon=False)

ordered = report.sort_values("shift_magnitude", ascending=True)
y = np.arange(len(ordered))
axes[1].barh(y, ordered["shift_magnitude"], color="#54A24B")
axes[1].errorbar(
    ordered["shift_magnitude"],
    y,
    xerr=np.vstack([(ordered["shift_magnitude"] - ordered["ci95_low"]).clip(lower=0), (ordered["ci95_high"] - ordered["shift_magnitude"]).clip(lower=0)]),
    fmt="none",
    color="#222222",
    capsize=3,
    linewidth=1,
)
axes[1].set_yticks(y, ordered["state"])
axes[1].set_xlabel("Centroid-shift magnitude")
axes[1].set_title("Descriptive perturbation ranking")

top = report.iloc[0]
alt = (
    f"Quickstart synthetic perturbation report. The generated dataset has {len(cells)} cells across {len(states)} states and two conditions. "
    f"The largest descriptive centroid shift is {top['state']} with magnitude {top['shift_magnitude']:.3f}; no threshold is selected or tuned."
)
artifacts = write_figure_bundle("tutorial_quickstart_perturbation_report", fig, report, alt)
artifacts

In [ ]:
metadata = write_metadata("01_quickstart_perturbation_report", {
    "n_cells": int(len(cells)),
    "n_states": int(len(states)),
    "n_bootstrap_resamples": 500,
    "figure": "tutorial_quickstart_perturbation_report",
})
metadata